In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.8.0+cu126
True


In [2]:
import os
import time
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

# -----------------------------
# Robust FASTA reader
# -----------------------------
try:
    from Bio import SeqIO
    BIO_OK = True
except Exception:
    BIO_OK = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# -----------------------------
# Device + seeds
# -----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}", flush=True)

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# -----------------------------
# Hyperparams
# -----------------------------
MAX_LEN     = 2000
MIN_FREQ    = 30
BATCH_SIZE  = 32
NUM_EPOCHS  = 3
LEARNING_RATE = 1e-3
ALPHA       = 0.5     # CNN vs similarity if DIAMOND used
TOP_K       = 200     # how many GO terms per protein in submission

# -----------------------------
# Amino acid vocabulary
# -----------------------------
AA_CHARS = 'ACDEFGHIKLMNPQRSTVWY'
aa2idx = {aa: i for i, aa in enumerate(AA_CHARS)}
aa2idx['X'] = 20  # unknown
NUM_AA = 21

# -----------------------------
# ID normalization
# -----------------------------
def normalize_protein_id(raw_id: str) -> str:
    """
    Normalize FASTA and TSV IDs to UniProt-like accessions.

    Examples:
    - 'sp|P9WHI7|RECN_MYCT' -> 'P9WHI7'
    - 'tr|A0A009IHW8|SOME_NAME' -> 'A0A009IHW8'
    - 'Q9CQV8|10090' -> 'Q9CQV8'
    - 'Q9CQV8' -> 'Q9CQV8'
    """
    if "|" in raw_id:
        parts = raw_id.split("|")
        if parts[0] in ("sp", "tr") and len(parts) >= 2:
            return parts[1]
        return parts[0]
    return raw_id

# -----------------------------
# FASTA loaders
# -----------------------------
def load_fasta_biopython(fasta_path):
    sequences = {}
    for record in SeqIO.parse(fasta_path, 'fasta'):
        prot_id = normalize_protein_id(record.id)
        sequences[prot_id] = str(record.seq)
    return sequences

def load_fasta_fallback(fasta_path):
    sequences = {}
    current_id = None
    current_seq = []
    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[normalize_protein_id(current_id)] = "".join(current_seq)
                header = line[1:].split()[0]
                current_id = header
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[normalize_protein_id(current_id)] = "".join(current_seq)
    return sequences

def load_fasta(fasta_path):
    if BIO_OK:
        return load_fasta_biopython(fasta_path)
    return load_fasta_fallback(fasta_path)

# -----------------------------
# Label loader
# -----------------------------
def load_train_labels(tsv_path):
    protein_go_map = defaultdict(list)
    with open(tsv_path, 'r') as f:
        for line_idx, line in enumerate(f):
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            raw_id = parts[0]

            # skip header-ish rows
            if raw_id.lower() in ("entryid", "protein_id", "entry_id"):
                continue

            protein_id = normalize_protein_id(raw_id)
            go_term = parts[1]
            if go_term and go_term.startswith("GO:"):
                protein_go_map[protein_id].append(go_term)
    return dict(protein_go_map)

# -----------------------------
# GO vocab builder
# -----------------------------
def build_go_vocabulary(protein_go_map, min_freq=MIN_FREQ):
    go_counter = Counter()
    for go_terms in protein_go_map.values():
        for go in go_terms:
            go_counter[go] += 1

    frequent_gos = [go for go, count in go_counter.items() if count >= min_freq]
    frequent_gos.sort()

    go2idx = {go: i for i, go in enumerate(frequent_gos)}
    idx2go = frequent_gos

    print(f"Total GO terms before filtering: {len(go_counter)}", flush=True)
    print(f"GO terms with >= {min_freq} occurrences: {len(frequent_gos)}", flush=True)

    return go2idx, idx2go

# -----------------------------
# Sequence encoding
# -----------------------------
def encode_sequence(seq, max_len=MAX_LEN):
    encoding = np.zeros((NUM_AA, max_len), dtype=np.float32)
    seq = seq[:max_len]
    for i, aa in enumerate(seq):
        aa_idx = aa2idx.get(aa.upper(), 20)
        encoding[aa_idx, i] = 1.0
    return encoding

# -----------------------------
# Datasets
# -----------------------------
class ProteinDataset(Dataset):
    def __init__(self, protein_ids, seq_dict, protein_go_map, go2idx, max_len=MAX_LEN):
        self.protein_ids = protein_ids
        self.seq_dict = seq_dict
        self.protein_go_map = protein_go_map
        self.go2idx = go2idx
        self.max_len = max_len
        self.num_go_terms = len(go2idx)

    def __len__(self):
        return len(self.protein_ids)

    def __getitem__(self, idx):
        protein_id = self.protein_ids[idx]
        seq = self.seq_dict[protein_id]
        encoded_seq = encode_sequence(seq, self.max_len)

        labels = np.zeros(self.num_go_terms, dtype=np.float32)
        for go_term in self.protein_go_map.get(protein_id, []):
            gi = self.go2idx.get(go_term)
            if gi is not None:
                labels[gi] = 1.0

        return torch.FloatTensor(encoded_seq), torch.FloatTensor(labels)

class TestProteinDataset(Dataset):
    def __init__(self, protein_ids, seq_dict, max_len=MAX_LEN):
        self.protein_ids = protein_ids
        self.seq_dict = seq_dict
        self.max_len = max_len

    def __len__(self):
        return len(self.protein_ids)

    def __getitem__(self, idx):
        protein_id = self.protein_ids[idx]
        seq = self.seq_dict[protein_id]
        encoded_seq = encode_sequence(seq, self.max_len)
        return torch.FloatTensor(encoded_seq), protein_id

# -----------------------------
# Model (DeepGOPlus style CNN)
# -----------------------------
class DeepGOPlusCNN(nn.Module):
    def __init__(self, num_go_terms):
        super().__init__()
        self.kernel_sizes = [8, 16, 24, 32, 40, 48, 56, 64]
        self.num_filters = 512

        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=NUM_AA, out_channels=self.num_filters, kernel_size=k)
            for k in self.kernel_sizes
        ])

        total_features = self.num_filters * len(self.kernel_sizes)
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(total_features, num_go_terms)

    def forward(self, x):
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = F.relu(conv(x))
            pooled = F.adaptive_max_pool1d(conv_out, 1).squeeze(-1)
            conv_outputs.append(pooled)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return self.fc(x)

# -----------------------------
# Training
# -----------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch_idx, num_epochs):
    model.train()
    total_loss = 0.0
    num_batches = len(dataloader)
    epoch_start = time.time()

    for batch_idx, (sequences, labels) in enumerate(dataloader, start=1):
        batch_start = time.time()

        sequences = sequences.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(sequences)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_time = time.time() - batch_start

        if batch_idx % 50 == 0 or batch_idx == num_batches:
            avg_loss = total_loss / batch_idx
            elapsed = time.time() - epoch_start
            avg_batch = elapsed / batch_idx
            eta = (num_batches - batch_idx) * avg_batch

            print(
                f"Epoch {epoch_idx+1}/{num_epochs} | "
                f"Batch {batch_idx}/{num_batches} | "
                f"Batch time: {batch_time:.3f}s | "
                f"Avg loss: {avg_loss:.4f} | "
                f"ETA: {eta/60:.1f} min",
                flush=True
            )

    return total_loss / num_batches

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for sequences, labels in dataloader:
            sequences = sequences.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(sequences)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            num_batches += 1

    return total_loss / max(1, num_batches)

def train_model(model, train_loader, val_loader, num_epochs, device):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    # FIXED: remove 'verbose' argument (not supported in this PyTorch)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )

    best_val_loss = float('inf')
    best_state = None

    print(f"Total batches per epoch: {len(train_loader)}", flush=True)

    for epoch in range(num_epochs):
        epoch_start = time.time()

        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, num_epochs)
        val_loss = validate(model, val_loader, criterion, device)

        scheduler.step(val_loss)
        epoch_time = time.time() - epoch_start

        print(
            f"Epoch {epoch+1}/{num_epochs} - "
            f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f} "
            f"({epoch_time/60:.1f} min)",
            flush=True
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  New best model (val_loss: {best_val_loss:.4f})", flush=True)

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# -----------------------------
# Prediction
# -----------------------------
def predict_test_set(model, test_loader, device):
    model.eval()
    predictions = {}
    num_batches = len(test_loader)
    start_time = time.time()

    with torch.no_grad():
        for batch_idx, (sequences, protein_ids) in enumerate(test_loader, start=1):
            batch_start = time.time()

            sequences = sequences.to(device, non_blocking=True)
            logits = model(sequences)
            probs = torch.sigmoid(logits).cpu().numpy()

            for i, pid in enumerate(protein_ids):
                predictions[pid] = probs[i]

            if batch_idx % 100 == 0 or batch_idx == num_batches:
                batch_time = time.time() - batch_start
                elapsed = time.time() - start_time
                avg_batch = elapsed / batch_idx
                eta = (num_batches - batch_idx) * avg_batch

                print(
                    f"Prediction | Batch {batch_idx}/{num_batches} | "
                    f"Batch time: {batch_time:.3f}s | ETA: {eta/60:.1f} min",
                    flush=True
                )

    return predictions

# -----------------------------
# Optional DIAMOND support
# -----------------------------
def load_diamond_hits(diamond_tsv_path):
    if not os.path.exists(diamond_tsv_path):
        return None
    df = pd.read_csv(diamond_tsv_path, sep='\t', names=['qseqid', 'sseqid', 'bitscore'])
    df['qseqid'] = df['qseqid'].apply(normalize_protein_id)
    df['sseqid'] = df['sseqid'].apply(normalize_protein_id)
    return df

def compute_similarity_scores(diamond_df, protein_go_map, go2idx, query_ids):
    similarity_matrix = np.zeros((len(query_ids), len(go2idx)), dtype=np.float32)
    query_id_to_idx = {qid: i for i, qid in enumerate(query_ids)}

    grouped = diamond_df.groupby('qseqid')

    for query_id, hits in grouped:
        qi = query_id_to_idx.get(query_id)
        if qi is None:
            continue

        total_bitscore = hits['bitscore'].sum()
        if total_bitscore == 0:
            continue

        go_bitscores = defaultdict(float)
        for _, row in hits.iterrows():
            sid = row['sseqid']
            b = row['bitscore']
            for go in protein_go_map.get(sid, []):
                gi = go2idx.get(go)
                if gi is not None:
                    go_bitscores[go] += b

        for go, gb in go_bitscores.items():
            gi = go2idx[go]
            similarity_matrix[qi, gi] = gb / total_bitscore

    return similarity_matrix

def combine_scores(cnn_scores, similarity_scores=None, alpha=0.5):
    if similarity_scores is None:
        return cnn_scores
    return alpha * cnn_scores + (1 - alpha) * similarity_scores

# -----------------------------
# TOP-K submission writer (NO sample file)
# -----------------------------
def create_topk_submission(final_predictions, go2idx, out_path, top_k=200):
    """
    final_predictions: dict[protein_id] -> np.array of shape (num_go_terms,)
    go2idx: dict[GO_ID] -> index
    Writes: EntryID, term, score with top_k GO terms per protein.
    NOTE: This writes a header line. If Kaggle expects NO HEADER,
    you can remove the first write() line.
    """
    idx2go = [""] * len(go2idx)
    for go, idx in go2idx.items():
        idx2go[idx] = go

    print(f"Writing submission to {out_path} (TOP_K={top_k}) ...", flush=True)
    num_proteins = len(final_predictions)

    with open(out_path, "w") as f:
        # If Kaggle complains, delete this header line:
        f.write("EntryID\tterm\tscore\n")
        for i, (pid, scores) in enumerate(final_predictions.items(), start=1):
            scores = np.asarray(scores, dtype=np.float32)
            if top_k < len(scores):
                top_idx = np.argpartition(-scores, top_k-1)[:top_k]
                top_idx = top_idx[np.argsort(-scores[top_idx])]
            else:
                top_idx = np.argsort(-scores)

            for idx in top_idx:
                go_id = idx2go[idx]
                s = float(scores[idx])
                f.write(f"{pid}\t{go_id}\t{s:.6f}\n")

            if i % 10000 == 0 or i == num_proteins:
                print(f"  Wrote {i}/{num_proteins} proteins", flush=True)

    print("Done writing submission.", flush=True)

# -----------------------------
# Main
# -----------------------------
def main():
    print("=" * 60, flush=True)
    print("CAFA 6 DeepGOPlus CNN Pipeline (TOP-K submission)", flush=True)
    print("=" * 60, flush=True)

    train_fasta = '/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta'
    train_terms = '/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv'
    test_fasta  = '/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta'
    output_submission = '/kaggle/working/submission.tsv'

    print("\n1. Loading sequences...", flush=True)
    train_seqs = load_fasta(train_fasta)
    print(f"Loaded {len(train_seqs)} training sequences", flush=True)

    test_seqs = load_fasta(test_fasta)
    print(f"Loaded {len(test_seqs)} test sequences", flush=True)

    print("\n2. Loading labels...", flush=True)
    protein_go_map = load_train_labels(train_terms)
    print(f"Loaded labels for {len(protein_go_map)} proteins", flush=True)

    print("\n3. Building GO vocabulary...", flush=True)
    go2idx, idx2go = build_go_vocabulary(protein_go_map, min_freq=MIN_FREQ)
    num_go_terms = len(go2idx)

    print("\n4. Preparing training data...", flush=True)
    train_protein_ids = [pid for pid in protein_go_map.keys() if pid in train_seqs]
    print(f"Proteins with both sequences and labels: {len(train_protein_ids)}", flush=True)

    if len(train_protein_ids) == 0:
        raise ValueError("No overlap between train labels and sequences. Check ID parsing.")

    train_ids, val_ids = train_test_split(train_protein_ids, test_size=0.1, random_state=42)
    print(f"Train set: {len(train_ids)}, Validation set: {len(val_ids)}", flush=True)

    train_dataset = ProteinDataset(train_ids, train_seqs, protein_go_map, go2idx)
    val_dataset   = ProteinDataset(val_ids, train_seqs, protein_go_map, go2idx)

    num_workers = 0  # safer on Kaggle

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )

    print("\n5. Building model...", flush=True)
    model = DeepGOPlusCNN(num_go_terms=num_go_terms).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}", flush=True)

    print("\n6. Training model...", flush=True)
    model = train_model(model, train_loader, val_loader, NUM_EPOCHS, device)

    print("\n7. Predicting on ALL test superset proteins...", flush=True)
    test_protein_ids = list(test_seqs.keys())
    print(f"Total real test proteins (superset): {len(test_protein_ids)}", flush=True)

    test_dataset = TestProteinDataset(test_protein_ids, test_seqs)
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )

    cnn_predictions = predict_test_set(model, test_loader, device)
    print(f"Generated CNN predictions for {len(cnn_predictions)} test proteins", flush=True)

    print("\n8. Loading DIAMOND hits (if available)...", flush=True)
    diamond_test_path = '/kaggle/working/diamond_test_hits.tsv'
    diamond_df = load_diamond_hits(diamond_test_path)

    if diamond_df is not None:
        print(f"Loaded {len(diamond_df)} DIAMOND hits (raw)", flush=True)
        diamond_df = diamond_df[diamond_df['qseqid'].isin(set(test_protein_ids))]
        print(f"Filtered to {len(diamond_df)} hits for test proteins", flush=True)

        print("Computing similarity scores...", flush=True)
        sim_matrix = compute_similarity_scores(diamond_df, protein_go_map, go2idx, test_protein_ids)

        print("Combining CNN + similarity scores...", flush=True)
        final_predictions = {}
        for i, pid in enumerate(test_protein_ids):
            final_predictions[pid] = combine_scores(cnn_predictions[pid], sim_matrix[i], alpha=ALPHA)

        print(f"Combined predictions with alpha={ALPHA}", flush=True)
    else:
        print("No DIAMOND hits found, using CNN predictions only", flush=True)
        final_predictions = cnn_predictions

    print("\n9. Creating TOP-K submission (NO sample file)...", flush=True)
    create_topk_submission(final_predictions, go2idx, output_submission, top_k=TOP_K)

    print("\n" + "=" * 60, flush=True)
    print("Pipeline complete!", flush=True)
    print("=" * 60, flush=True)

if __name__ == '__main__':
    main()

Using device: cuda
CAFA 6 DeepGOPlus CNN Pipeline (TOP-K submission)

1. Loading sequences...
Loaded 82404 training sequences
Loaded 224309 test sequences

2. Loading labels...
Loaded labels for 82404 proteins

3. Building GO vocabulary...
Total GO terms before filtering: 26125
GO terms with >= 30 occurrences: 2814

4. Preparing training data...
Proteins with both sequences and labels: 82404
Train set: 74163, Validation set: 8241

5. Building model...
Total parameters: 14,629,630

6. Training model...
Total batches per epoch: 2318
Epoch 1/3 | Batch 50/2318 | Batch time: 0.125s | Avg loss: 0.0363 | ETA: 5.6 min
Epoch 1/3 | Batch 100/2318 | Batch time: 0.125s | Avg loss: 0.0237 | ETA: 5.2 min
Epoch 1/3 | Batch 150/2318 | Batch time: 0.125s | Avg loss: 0.0197 | ETA: 5.0 min
Epoch 1/3 | Batch 200/2318 | Batch time: 0.125s | Avg loss: 0.0175 | ETA: 4.8 min
Epoch 1/3 | Batch 250/2318 | Batch time: 0.125s | Avg loss: 0.0161 | ETA: 4.7 min
Epoch 1/3 | Batch 300/2318 | Batch time: 0.125s | Avg 